In [1]:
!pip install torch pytorch-lightning transformers datasets scikit-learn wandb

In [2]:
!pip install -U datasets

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger
from transformers import AutoTokenizer, AutoModel, AutoConfig
from datasets import load_dataset
import numpy as np
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
import wandb
import warnings
warnings.filterwarnings('ignore')

In [4]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    labels = [item['labels'] for item in batch]

    # Pad sequences to the maximum length in the batch
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100) # Pad labels with -100

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [5]:
class NERDataset(Dataset):
    def __init__(self, tokenized_dataset):
        self.tokenized_dataset = tokenized_dataset

    def __getitem__(self, idx):
        item = self.tokenized_dataset[idx]
        return {
            'input_ids': torch.tensor(item['input_ids']),
            'attention_mask': torch.tensor(item['attention_mask']),
            'labels': torch.tensor(item['labels'])
        }

    def __len__(self):
        return len(self.tokenized_dataset)

In [6]:
class NERDataModule(pl.LightningDataModule):
    def __init__(self, model_name='microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract', batch_size=16, max_length=128):
        super().__init__()
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Load BC5CDR dataset (biomedical NER for chemicals and diseases)
        self.dataset = load_dataset("tner/bc5cdr")

        # BC5CDR uses BIO tagging scheme for chemicals and diseases
        # Based on the dataset structure: 0=O, 1=B-Chemical, 2=I-Chemical, 3=B-Disease, 4=I-Disease
        self.label_names = ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']
        self.label2id = {label: i for i, label in enumerate(self.label_names)}
        self.id2label = {i: label for i, label in enumerate(self.label_names)}
        self.num_labels = len(self.label_names)

        print(f"Dataset loaded with {len(self.label_names)} labels: {self.label_names}")

    def tokenize_and_align_labels(self, examples):
        tokenized_inputs = self.tokenizer(
            examples['tokens'],
            truncation=True,
            is_split_into_words=True,
            padding=False,  # We'll handle padding in collate_fn
            max_length=self.max_length
        )

        labels = []
        for i, label in enumerate(examples['tags']):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # Special token
                elif word_idx != previous_word_idx:
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)  # Subword token
                previous_word_idx = word_idx

            labels.append(label_ids)

        tokenized_inputs['labels'] = labels
        return tokenized_inputs

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            train_dataset = self.dataset['train']
            val_dataset = self.dataset['validation']

            train_tokenized = train_dataset.map(
                self.tokenize_and_align_labels,
                batched=True,
                remove_columns=train_dataset.column_names
            )

            val_tokenized = val_dataset.map(
                self.tokenize_and_align_labels,
                batched=True,
                remove_columns=val_dataset.column_names
            )

            self.train_dataset = NERDataset(train_tokenized)
            self.val_dataset = NERDataset(val_tokenized)

        if stage == 'test' or stage is None:
            test_dataset = self.dataset['test']
            test_tokenized = test_dataset.map(
                self.tokenize_and_align_labels,
                batched=True,
                remove_columns=test_dataset.column_names
            )
            self.test_dataset = NERDataset(test_tokenized)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, collate_fn=collate_fn)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, collate_fn=collate_fn)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, collate_fn=collate_fn)

In [7]:
class CustomTransformer(nn.Module):
    """Custom transformer architecture for comparison"""
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=6, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(max_seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids, attention_mask=None):
        seq_len = input_ids.size(1)

        # Embeddings
        x = self.embedding(input_ids) * np.sqrt(self.d_model)
        x = x + self.pos_embedding[:seq_len, :]

        # Create attention mask for transformer
        if attention_mask is not None:
            # Convert to boolean mask (True for positions to attend to)
            mask = attention_mask.bool()
            # Invert for transformer (True for positions to ignore)
            mask = ~mask
        else:
            mask = None

        # Apply transformer
        x = self.transformer(x, src_key_padding_mask=mask)
        x = self.layer_norm(x)

        return type('TransformerOutput', (), {'last_hidden_state': x})()

In [14]:
class NERModel(pl.LightningModule):
    def __init__(self, model_name='microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract',
                 num_labels=9, learning_rate=2e-5, dropout=0.1, use_custom_transformer=False,
                 label_names=None, tokenizer=None):
        super().__init__()
        self.save_hyperparameters()

        self.model_name = model_name
        self.num_labels = num_labels
        self.learning_rate = learning_rate
        self.use_custom_transformer = use_custom_transformer
        self.label_names = label_names or []
        self.tokenizer = tokenizer

        # Model architecture
        if use_custom_transformer:
            # Custom transformer
            vocab_size = 30522  # Default BERT vocab size
            if tokenizer:
                vocab_size = len(tokenizer.get_vocab())
            self.transformer = CustomTransformer(vocab_size=vocab_size)
            hidden_size = 256
        else:
            # Pre-trained transformer
            self.transformer = AutoModel.from_pretrained(model_name)
            hidden_size = self.transformer.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

        # Loss function
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

        # For tracking metrics
        self.validation_outputs = []

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))

        return {'loss': loss, 'logits': logits}

    def training_step(self, batch, batch_idx):
        outputs = self.forward(**batch)
        loss = outputs['loss']
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self.forward(**batch)
        loss = outputs['loss']
        logits = outputs['logits']

        # Calculate predictions
        predictions = torch.argmax(logits, dim=-1)

        # Store for epoch-end processing
        self.validation_outputs.append({
            'loss': loss,
            'predictions': predictions,
            'labels': batch['labels'],
            'input_ids': batch['input_ids'],
            'attention_mask': batch['attention_mask']
        })

        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def on_validation_epoch_end(self):
        # Aggregate all validation outputs
        all_predictions = []
        all_labels = []

        for output in self.validation_outputs:
            predictions = output['predictions']
            labels = output['labels']

            # Flatten predictions and labels, excluding ignored tokens
            flat_predictions = predictions.view(-1)
            flat_labels = labels.view(-1)

            # Remove ignored tokens
            valid_mask = flat_labels != -100
            valid_predictions = flat_predictions[valid_mask]
            valid_labels = flat_labels[valid_mask]

            all_predictions.extend(valid_predictions.cpu().numpy())
            all_labels.extend(valid_labels.cpu().numpy())

        # Calculate overall metrics
        f1 = f1_score(all_labels, all_predictions, average='weighted')
        precision, recall, _, _ = precision_recall_fscore_support(
            all_labels, all_predictions, average='weighted'
        )

        # Calculate per-entity metrics
        entity_metrics = self.calculate_entity_metrics(all_labels, all_predictions)

        # Log metrics to wandb
        metrics = {
            'val_f1': f1,
            'val_precision': precision,
            'val_recall': recall,
            **entity_metrics
        }

        self.log_dict(metrics, on_epoch=True)

        # Log text examples with predictions
        self.log_text_examples()

        # Clear validation outputs for next epoch
        self.validation_outputs = []

    def calculate_entity_metrics(self, labels, predictions):
        """Calculate metrics for each entity type"""
        entity_metrics = {}

        for i, label_name in enumerate(self.label_names):
            if label_name == 'O':  # Skip 'Outside' label
                continue

            # Binary classification for this entity
            true_binary = [1 if l == i else 0 for l in labels]
            pred_binary = [1 if p == i else 0 for p in predictions]

            if sum(true_binary) > 0 or sum(pred_binary) > 0:  # Only if entity exists
                precision, recall, f1, _ = precision_recall_fscore_support(
                    true_binary, pred_binary, average='binary', zero_division=0
                )

                entity_metrics[f'val_{label_name}_precision'] = precision
                entity_metrics[f'val_{label_name}_recall'] = recall
                entity_metrics[f'val_{label_name}_f1'] = f1

        return entity_metrics

    def log_text_examples(self):
        """Log text examples with predicted entities to wandb"""
        if not self.tokenizer or not self.validation_outputs:
            return

        # Take first batch for examples
        batch = self.validation_outputs[0]
        input_ids = batch['input_ids'][:3]  # First 3 examples
        predictions = batch['predictions'][:3]
        labels = batch['labels'][:3]
        attention_mask = batch['attention_mask'][:3]

        examples = []
        for i in range(len(input_ids)):
            # Decode tokens
            tokens = self.tokenizer.convert_ids_to_tokens(input_ids[i])
            pred_labels = predictions[i]
            true_labels = labels[i]
            mask = attention_mask[i]

            # Create example text with entities
            example_data = []
            for j, (token, pred_label, true_label, attention) in enumerate(
                zip(tokens, pred_labels, true_labels, mask)
            ):
                if attention == 1 and token not in ['[CLS]', '[SEP]', '[PAD]']:
                    pred_entity = self.label_names[pred_label.item()] if pred_label.item() < len(self.label_names) else 'UNK'
                    true_entity = self.label_names[true_label.item()] if true_label.item() != -100 and true_label.item() < len(self.label_names) else 'O'

                    example_data.append({
                        'token': token,
                        'predicted': pred_entity,
                        'true': true_entity,
                        'correct': pred_entity == true_entity
                    })

            examples.append(example_data)

        # Log to wandb as table
        if examples:
            columns = ['token', 'predicted', 'true', 'correct']
            for i, example in enumerate(examples):
                table_data = [[row[col] for col in columns] for row in example]
                table = wandb.Table(data=table_data, columns=columns)
                wandb.log({f'example_{i+1}': table})

    def test_step(self, batch, batch_idx):
        outputs = self.forward(**batch)
        loss = outputs['loss']
        logits = outputs['logits']

        predictions = torch.argmax(logits, dim=-1)

        # Flatten predictions and labels, excluding ignored tokens
        flat_predictions = predictions.view(-1)
        flat_labels = batch['labels'].view(-1)

        # Remove ignored tokens
        valid_mask = flat_labels != -100
        valid_predictions = flat_predictions[valid_mask]
        valid_labels = flat_labels[valid_mask]

        return {
            'test_loss': loss,
            'predictions': valid_predictions.cpu(),
            'labels': valid_labels.cpu()
        }

    def configure_optimizers(self):
        if self.use_custom_transformer:
            # Higher learning rate for custom transformer
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate * 10)
        else:
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)

        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
        return [optimizer], [scheduler]

In [15]:
def train_ner_model(use_custom_transformer=False, experiment_name="biomedical_ner",
                    epoch_count = 10):
    # Initialize wandb
    wandb.init(
        project="biomedical-ner-comparison",
        name=f"{experiment_name}_{'custom' if use_custom_transformer else 'pretrained'}",
        config={
            "model_type": "custom_transformer" if use_custom_transformer else "pretrained",
            "dataset": "bc5cdr",
            "batch_size": 16,
            "learning_rate": 2e-5,
            "max_epochs": 10
        }
    )

    # Initialize data module
    data_module = NERDataModule(
        model_name='microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract',
        batch_size=16,
        max_length=128
    )

    # Setup data
    data_module.setup()

    # Initialize model
    model = NERModel(
        model_name='microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract',
        num_labels=data_module.num_labels,
        learning_rate=2e-5,
        dropout=0.1,
        use_custom_transformer=use_custom_transformer,
        label_names=data_module.label_names,
        tokenizer=data_module.tokenizer
    )

    # Callbacks
    checkpoint_callback = ModelCheckpoint(
        monitor='val_f1',
        mode='max',
        save_top_k=1,
        filename=f'best-ner-model-{experiment_name}-{{epoch:02d}}-{{val_f1:.3f}}'
    )

    early_stopping = EarlyStopping(
        monitor='val_f1',
        mode='max',
        patience=3,
        verbose=True
    )

    # WandB Logger
    wandb_logger = WandbLogger(
        project="biomedical-ner-comparison",
        name=f"{experiment_name}_{'custom' if use_custom_transformer else 'pretrained'}"
    )

    # Trainer
    trainer = pl.Trainer(
        max_epochs=epoch_count,
        accelerator='auto',
        devices=1,
        callbacks=[checkpoint_callback, early_stopping],
        logger=wandb_logger,
        log_every_n_steps=50
    )

    # Train model
    trainer.fit(model, data_module)

    # Test model
    trainer.test(model, data_module)

    # Close wandb run
    wandb.finish()

    return model, trainer

In [16]:
def predict_ner(model, tokenizer, text, label_names):
    """Function to predict NER tags for biomedical text"""
    model.eval()

    # Tokenize input
    inputs = tokenizer(
        text.split(),
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs['logits'], dim=-1)

    # Align predictions with tokens
    word_ids = inputs.word_ids()
    previous_word_idx = None
    predicted_labels = []
    tokens = text.split()

    for i, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != previous_word_idx:
            predicted_labels.append(label_names[predictions[0][i].item()])
            previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))


In [17]:
model_pretrained, trainer_pretrained = train_ner_model(
        use_custom_transformer=False,
        experiment_name="pubmedbert",
        epoch_count=3
    )

epoch,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇█████
train_loss_epoch,█▄▃▃▂▁▁▁▁▁
train_loss_step,█▆▅▆▄▃▃▃▃▃▂▃▃▂▂▂▁▁▂▂▁▁▂▂▁▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
val_B-Chemical_f1,▁▆▆▇▇█████
val_B-Chemical_precision,▁▆▃▃▄█▇███
val_B-Chemical_recall,▁▅███▇████
val_B-Disease_f1,▁▂▅▆▇▇████
val_B-Disease_precision,█▆▄▁▄▄▄▅▃▅
val_B-Disease_recall,▁▂▅▇▇▇█▇█▇
val_I-Chemical_f1,▁▅▇▇▇█████


Dataset loaded with 5 labels: ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']


Map:   0%|          | 0/5865 [00:00<?, ? examples/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Map:   0%|          | 0/5228 [00:00<?, ? examples/s]

...

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_f1 improved by 0.001 >= min_delta = 0.0. New best score: 0.979
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.


Map:   0%|          | 0/5865 [00:00<?, ? examples/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

epoch,▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅████████
train_loss_epoch,█▂▁
train_loss_step,█▄▅▅▃▂▂▁▁▄▂▃▁▂▁▁▁▁▁
trainer/global_step,▁▁▂▂▃▃▃▃▃▄▄▄▅▅▆▆▆▆▆▇▇▇███
val_B-Chemical_f1,█▁█
val_B-Chemical_precision,▂▁█
val_B-Chemical_recall,██▁
val_B-Disease_f1,▅█▁
val_B-Disease_precision,▁▃█
val_B-Disease_recall,█▆▁
val_I-Chemical_f1,▁██


In [18]:
model_custom, trainer_custom = train_ner_model(
        use_custom_transformer=True,
        experiment_name="custom_transformer"
    )

Dataset loaded with 5 labels: ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Map:   0%|          | 0/5228 [00:00<?, ? examples/s]

Map:   0%|          | 0/5330 [00:00<?, ? examples/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name        | Type              | Params | Mode 
----------------------------------------------------------
0 | transformer | CustomTransformer | 12.2 M | train
1 | dropout     | Dropout           | 0      | train
2 | classifier  | Linear            | 1.3 K  | train
3 | loss_fn     | CrossEntropyLoss  | 0      | train
----------------------------------------------------------
12.2 M    Trainable params
0         Non-trainable params
12.2 M    Total params
48.681    Total estimated model params size (MB)
68        Modules in train mode
0         Modules in eval mode


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Map:   0%|          | 0/5865 [00:00<?, ? examples/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

epoch,▁▁▁▁▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇████
train_loss_epoch,█▅▃▃▂▁▁▁▁▁
train_loss_step,█▆▇▆▆▆▃▇▃▄▃▃▂▃▂▂▂▃▂▃▂▂▂▃▂▂▂▁▂▂▁▁▁▁▂▁▁▂▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇████
val_B-Chemical_f1,▁▅▇▇██████
val_B-Chemical_precision,▁█▅▂▇▇▆▇▇▇
val_B-Chemical_recall,▁▃▇█▇▆▇▇▇▇
val_B-Disease_f1,▁▂▆▇▇▇████
val_B-Disease_precision,█▇▁▄▇▇▅▅▆▆
val_B-Disease_recall,▁▂█▆▆▆█▇▇▇
val_I-Chemical_f1,▁▄▆▇▇█████
